In [1]:
import sys
import os
import json
from pathlib import Path

# 1. Path Setup
_PROJECT_ROOT = Path.cwd().parents[2]
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

_RQ1_DIR = Path.cwd().parent
if str(_RQ1_DIR) not in sys.path:
    sys.path.insert(0, str(_RQ1_DIR))

from evaluate_rq1 import *
print(f"✓ Success! Project root set to: {_PROJECT_ROOT}")

# 2. Define File Paths
# Point this directly to your test file
TEST_DATA_PATH = os.path.join(_PROJECT_ROOT, "datasets", "wikidata_testset_RQ1.json") 
CHECKPOINT_PATH = os.path.join(_PROJECT_ROOT, "sparqlce", "models", "wikidata_model_mrr_0.3238.pt") 

TOPKS = [3, 5, 10, 20]
BATCH = 32

# 3. Load the test dataset directly
test_data = load_triplets(TEST_DATA_PATH)

print(f"Dataset Loaded -> Test Split: {len(test_data)}")

# 4. Build corpus and eval items using ONLY the test data
global_corpus, seen_q = build_global_corpus(test_data)
eval_items = build_eval_items(test_data, global_corpus, seen_q)
relevance_scores_per_item = build_binary_relevance_maps(eval_items, global_corpus)

print(f"Evaluation Ready -> Anchors: {len(eval_items)} | Corpus: {len(global_corpus)}")

/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Success! Project root set to: /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings
Dataset Loaded -> Test Split: 186
Evaluation Ready -> Anchors: 186 | Corpus: 2229


In [2]:
# SPARQLCE
evaluate_sparqlce_checkpoint(CHECKPOINT_PATH, eval_items, global_corpus, relevance_scores_per_item, TOPKS, batch_size=BATCH)

# TF-IDF
evaluate_tfidf(eval_items, global_corpus, relevance_scores_per_item, TOPKS)

# E5
evaluate_sentence_transformer('intfloat/e5-base-v2', 'E5-base-v2', eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=lambda q: f'query: {q}')

# Qwen
#evaluate_sentence_transformer('Qwen/Qwen3-Embedding-0.6B', 'Qwen3-Emb-0.6B', eval_items, global_corpus, relevance_scores_per_item, TOPKS)

Loading model from /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings/sparqlce/models/wikidata_model_mrr_0.3238.pt...

SPARQLCE
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1649  0.0383  0.0589  0.1998  0.3172    0.1649  0.1522
@5     0.1882  0.0751  0.0996  0.2294  0.4462    0.1882  0.1726
@10    0.1919  0.1549  0.1561  0.2429  0.5430    0.1919  0.1933
@20    0.1890  0.2937  0.2106  0.2504  0.6452    0.1890  0.2449

Baseline: TF-IDF
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0717  0.0143  0.0227  0.0726  0.0914    0.0717  0.0690
@5     0.0796  0.0263  0.0376  0.0804  0.1290    0.0796  0.0754
@10    0.0683  0.0453  0.0516  0.0847  0.1613    0.0683  0.0722
@20    0.0417  0.0621  0.0460  0.0880  0.2097    0.0417  0.061

In [3]:
display_name = "SBERT/MiniLM" 
model_id = "sentence-transformers/all-MiniLM-L6-v2"
evaluate_sentence_transformer(model_id, display_name, eval_items, global_corpus, relevance_scores_per_item, TOPKS)



SBERT/MiniLM
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0986  0.0197  0.0310  0.1039  0.1344    0.0986  0.0946
@5     0.1011  0.0330  0.0469  0.1077  0.1505    0.1011  0.0980
@10    0.0855  0.0568  0.0639  0.1157  0.2043    0.0855  0.0926
@20    0.0481  0.0630  0.0507  0.1176  0.2366    0.0481  0.0729


In [4]:
display_name = "SBERT/MPNet"
model_id = "sentence-transformers/all-mpnet-base-v2"
evaluate_sentence_transformer(model_id, display_name, eval_items, global_corpus, relevance_scores_per_item, TOPKS)



SBERT/MPNet
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0914  0.0172  0.0280  0.0986  0.1237    0.0914  0.0889
@5     0.0968  0.0326  0.0465  0.1021  0.1398    0.0968  0.0931
@10    0.0866  0.0575  0.0654  0.1109  0.2097    0.0866  0.0911
@20    0.0548  0.0722  0.0589  0.1141  0.2581    0.0548  0.0771
